In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv


In [2]:
from pathlib import Path
from sklearn.utils import shuffle

data_dir = Path('/kaggle/input/playground-series-s6e1/')
df_train = pd.read_csv(data_dir / 'train.csv')
df_test = pd.read_csv(data_dir / 'test.csv')

df_train = shuffle(df_train, random_state=42).reset_index(drop=True)

In [3]:
df_train

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,364426,18,other,b.sc,3.07,88.3,yes,4.1,average,coaching,low,hard,51.3
1,224752,21,male,b.com,3.28,49.6,no,7.0,poor,online videos,high,moderate,50.6
2,110423,20,female,bca,4.45,42.5,yes,6.7,good,group study,high,moderate,79.9
3,272555,21,male,b.com,4.19,82.8,yes,5.7,poor,mixed,low,hard,55.4
4,199651,21,female,b.com,1.31,91.2,yes,8.5,poor,mixed,low,easy,50.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,110268,21,male,bba,2.88,93.4,yes,6.1,poor,self-study,low,moderate,38.0
629996,259178,20,male,bca,6.90,74.5,yes,4.3,average,online videos,medium,moderate,70.1
629997,365838,20,other,b.com,1.54,59.3,yes,8.4,good,mixed,low,hard,39.9
629998,131932,20,other,b.tech,6.30,88.4,yes,6.2,poor,group study,low,moderate,69.3


In [4]:
df_train.dtypes

id                    int64
age                   int64
gender               object
course               object
study_hours         float64
class_attendance    float64
internet_access      object
sleep_hours         float64
sleep_quality        object
study_method         object
facility_rating      object
exam_difficulty      object
exam_score          float64
dtype: object

In [5]:
df_train.isna().sum()

id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [6]:
df_train.nunique()

id                  630000
age                      8
gender                   3
course                   7
study_hours            792
class_attendance       617
internet_access          2
sleep_hours             66
sleep_quality            3
study_method             5
facility_rating          3
exam_difficulty          3
exam_score             805
dtype: int64

In [7]:
cat_cols = df_train.select_dtypes(exclude=np.number).columns.tolist()

for col in cat_cols:
    print(f'Col "{col}" \t| {df_train[col].unique()}')

Col "gender" 	| ['other' 'male' 'female']
Col "course" 	| ['b.sc' 'b.com' 'bca' 'b.tech' 'bba' 'diploma' 'ba']
Col "internet_access" 	| ['yes' 'no']
Col "sleep_quality" 	| ['average' 'poor' 'good']
Col "study_method" 	| ['coaching' 'online videos' 'group study' 'mixed' 'self-study']
Col "facility_rating" 	| ['low' 'high' 'medium']
Col "exam_difficulty" 	| ['hard' 'moderate' 'easy']


In [8]:
df_train.select_dtypes(include=np.number).corr()

,id,age,study_hours,class_attendance,sleep_hours,exam_score
id,1.000000,-0.000581,0.000346,0.000677,0.001416,0.000372
age,-0.000581,1.000000,0.007545,0.005628,0.005864,0.010472
study_hours,0.000346,0.007545,1.000000,0.087617,0.042491,0.762267
class_attendance,0.000677,0.005628,0.087617,1.000000,0.029263,0.360954
sleep_hours,0.001416,0.005864,0.042491,0.029263,1.000000,0.167410
exam_score,0.000372,0.010472,0.762267,0.360954,0.167410,1.000000


In [9]:
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, StandardScaler

scaler = StandardScaler()
onehot = OneHotEncoder(sparse_output=False)

def preprocess(X, scaler, onehot, fit=True):
    if 'exam_score' in X.columns.tolist():
        X = X.drop(columns=[target, 'id'])
    else:
        X = X.drop(columns=['id'])

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    
    if fit:
        scaled = scaler.fit_transform(X[num_cols])
    else:
        scaled = scaler.transform(X[num_cols])
        
    X = X.drop(columns=num_cols)
    X[num_cols] = scaled
    
    cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

    if fit:
        encoded = onehot.fit_transform(X[cat_cols])
    else:
        encoded = onehot.transform(X[cat_cols])
    
    new_cols = onehot.get_feature_names_out()

    X = X.drop(columns=cat_cols)
    X[new_cols] = encoded

    return X


In [10]:
target = 'exam_score'
X_train, X_valid = preprocess(df_train[:600_000], scaler, onehot), preprocess(df_train[600_000:], scaler, onehot, fit=False)
y_train, y_valid = df_train[target][:600_000], df_train[target][600_000:]

In [11]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb
# param_grid = {
#     'max_depth': [3, 5, 7],
#     'learning_rate': [0.1, 0.01, 0.05],
#     'n_estimators': [50, 100, 200]
# }

# param_grid_lgb = {
#     'n_estimators': [50, 100, 200],
#     'learning_rate': [0.01, 0.05, 0.1],
#     'num_leaves': [31, 50, 100],
#     'max_depth': [5, 10, -1],
# }

# model = lgb.LGBMRegressor(boosting_type='gbdt', objective='regression', random_state=42, device='gpu')
# clf = GridSearchCV(
#     estimator=model,
#     param_grid=param_grid_lgb,
#     scoring='neg_root_mean_squared_error',
#     cv=3,
#     n_jobs=1,
#     verbose=2
# )

# model = XGBRegressor(random_state=42, objective='reg:squarederror', device='cuda')
# clf = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=1, verbose=3)

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [12]:
# clf.fit(X_train[:100_000], y_train[:100_000])
# print(f"Best score: {clf.best_score_:.4f}")
# print(f"Best params: {clf.best_params_}")

NameError: name 'clf' is not defined

In [ ]:
# Best score: 0.7831
# Best params: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 200}

# LGBM {'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}

# model = XGBRegressor(random_state=42, objective='reg:squarederror', learning_rate=0.05, max_depth=7, n_estimators=200)
# model.fit(X_train, y_train)

In [ ]:
# df['study_hoursXclass_attendance'] = df['study_hours'] * df['class_attendance']
# df['study_hoursOVERsleep_hours'] = df['study_hours'] / df['sleep_hours']
# df

In [13]:
from sklearn.metrics import root_mean_squared_error

# xgb_params = {
#     'learning_rate': 0.05,
#     'max_depth': 7,
#     'n_estimators': 200
# }

# model = XGBRegressor(random_state=42, objective='reg:squarederror', **xgb_params, device='cuda')
# model.fit(X_train, y_train)

# preds = model.predict(X_valid)
# rmse = root_mean_squared_error(y_valid, preds)

lgbm_params = {
    'learning_rate': 0.05,
    'max_depth': 10,
    'n_estimators': 200,
    'num_leaves': 31
}

model = lgb.LGBMRegressor(boosting_type='gbdt', objective='regression', random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_valid)
rmse = root_mean_squared_error(y_valid, preds)

print(rmse)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017333 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 625
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 30
[LightGBM] [Info] Start training from score 62.506526
8.874118680313265


In [14]:
X_train_full = preprocess(df_train, scaler, onehot)
y_train_full = df_train[target]
model.fit(X_train_full, y_train_full)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017473 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 625
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 30
[LightGBM] [Info] Start training from score 62.506672


LGBMRegressor(objective='regression', random_state=42)

In [15]:
X_test = preprocess(df_test, scaler, onehot, fit=False)

test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': df_test['id'],
    'exam_score': test_preds
})

submission.to_csv('submission.csv', index=False)